# ROOT TH2 efficiency and background maps

This notebook demonstrates ordinary-Dalitz TH2 maps using the same paper-inspired $B^+\to K^+\pi^+\pi^-$ benchmark as the main tutorials: $K^*(892)^0+(K\pi)_S+\rho(770)^0+f_0(980)+NR$. The ROOT loaders return fitter-ready efficiency/background objects that can be passed directly to `generate_toy` and `FitSession`.


In [ ]:
import numpy as np
import uproot
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    BaBarFlatte, BackgroundSpec, DecayChannel, DecayModel, FitSession, LASS,
    NonResonant, Parameter, RealImag, Resonance, ToyBackground, enable_x64,
    generate_toy, histogram_background_from_root, histogram_efficiency_from_root,
    plot_dalitz,
)
enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS": (1.40, -0.60),
    "rho770": (0.65, 0.10),
    "f0_980": (-0.20, 1.00),
    "NR": (-0.50, 0.10),
}
truth = {}

def coefficient(name, fixed=False):
    x, y = truth_xy[name]
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"] = x
    truth[f"{name}.y"] = y
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "Kstar892")) for name in truth_xy}
model = DecayModel(
    channel,
    [
        Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
        NonResonant(c["NR"]),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=300,
    normalization_pair=(0,2),
)


In [ ]:
x = np.linspace(0.3, 27.0, 31)
y = np.linspace(0.08, 23.0, 31)
xc = 0.5*(x[:-1]+x[1:]); yc = 0.5*(y[:-1]+y[1:])
X,Y = np.meshgrid(xc,yc,indexing="ij")
eff_values = 0.45 + 0.40*(X-X.min())/(X.max()-X.min())
bkg_values = 0.25 + 0.75*(Y-Y.min())/(Y.max()-Y.min())

with uproot.recreate("maps_dp.root") as f:
    f["efficiency_s13_s23"] = (eff_values,x,y)
    f["background_s13_s23"] = (bkg_values,x,y)

eff = histogram_efficiency_from_root("maps_dp.root","efficiency_s13_s23",x_variable="s13",y_variable="s23")
bkg = histogram_background_from_root("maps_dp.root","background_s13_s23",x_variable="s13",y_variable="s23")


In [ ]:
f_sig = Parameter("signal_fraction",0.75,bounds=(0.05,0.99))
data = generate_toy(
    model, 25_000, parameters=truth, efficiency=eff, signal_fraction=0.80,
    backgrounds=(ToyBackground("comb",bkg),), seed=1414, pool_size=220_000,
)
plot_dalitz(data,x="s13",y="s23",title="ROOT TH2 efficiency + background toy")
plt.show()

session = FitSession(model,data,efficiency=eff,signal_fraction=f_sig, backgrounds=(BackgroundSpec("comb",bkg),))
start = {p.name: truth[p.name] + 0.08 for p in session.parameters if p.name in truth}
start["signal_fraction"] = 0.72
result = session.fit(start, simplex=True, ncall=50_000)
session.report(result, acceptance_weighted_fractions=True)
session.plot_projection(result,"s13")
plt.show()
